# Exercise 3: Logistic Regression and Evaluation of Binary Classifiers

### Guidelines for Theoretical Exercises

* Write solutions in a file named  `hw3.pdf`.
* Show your work clearly and provide the intermediate steps.
* You may use Python to carry out numerical calculations.

### Guidelines for Programming Exercises

* Complete the required functions in `hw3.py`.
* All functions must operate only on the arguments they receive.
* Do not rename or change function signatures.
* Write efficient, vectorized code whenever possible. No individual cell should have a runtime of more than 5 minutes on a standard architecture. 
* You are allowed to use functions and methods from the Python standard library, `numpy`, and `pandas` only.
* Your code must run without errors.
* The file will be imported directly by the auto-grader.
* Do not execute code at import time.
* Do not read or write any files inside `hw3.py`. 
* This notebook is for guidance and self-testing only. You may add code and text to it, but it will not be tested.


### Submission Guidelines

* Submit a zip file that contains:
    - `hw3.py`
    - `hw3.ipynb` with your IDs in the signature line below and any tests you added (will not be checked)
    - `hw3.pdf`
* Use a zip filename that contains your ID(s), for example `hw3_123456789.zip` or `hw3_123456789_987654321.zip`.
* Place the files at the **top level of the zip**, not inside nested folders.
  

---
---

## Please sign that you have read and understood the instructions:

### *** YOUR IDs HERE ***

# ID1: 206630915
# ID2: 208004416


# Part 1: Theoretical Exercise (18 pts)

In class, we developed the logistic regression model by requiring that the labeling probability for the positive class
($\hat{y}_{w}(x) = \Pr[y=1|x,w]$) satisfies the log-odds condition:
$$
\ln\left[\frac{\hat{y}_{w}(x)}{1-\hat{y}_{w}(x)}\right] ~~=~~ w^\top x ~~=~~ w_0 + \sum_{i=1}^{p} w_i x_i ~.
$$

In class we showed that this assumption implies the following closed form expression for $\hat{y}_{w}(x)$:
$$
\hat{y}_{w}(x) ~~=~~ \sigma(w^\top x) ~~=~~ \frac{1}{1+ e^{-w^\top x}}
$$
Using this function for the class labeling probability, we defined the binary cross-entropy (BCE) loss and developed an expression for the gradient of the loss to use in a gradient descent algorithm.

In this question, we will derive a similar process for complementary log-log (CLL) regression.
In CLL, we make the following assumption about the labeling probability for the positive class:
$$
\ln( - \ln( 1 - \hat{y}_{w}(x))) ~~=~~ w^\top x ~~=~~ w_0 + \sum_{i=1}^{p} w_i x_i 
$$

1. Find an expression for $\hat{y}_{w}(x)$ as a function of $w^\top x$ that satisfies the CLL condition above. 
Write the expression as $\hat{y}_{w}(x)=\gamma(w^\top x)$ for some function $\gamma(t)$ that you need to find. This $\gamma(t)$ is the CLL analogue of LoR's $\sigma(t)$. 
2. Plot the function $\gamma(t)$ you derived in (1) in the range $t\in [-3,3]$. Also plot the logistic function $\sigma(t)$ in the same ragne to compare them.
3. Use the expression you obtained for $\gamma$ in (1) to compute the predicted probability $\hat{y}_{w}(x)$ under CLL when $w^\top x=0$.
4. Use the expression you obtained for $\gamma$ in (1) to compute $w^\top x$ that implies  $\hat{y}_{w}(x)=0.5$ under CLL. 
5. Use the expression you obtained for $\gamma$ in (1) to express the BCE loss as a function of $w^\top x$.
6. Express the gradient of the BCE loss and <u>describe the gradient descent procedure</u> for minimizing the BCE loss under CLL.<br> When computing an expression for the gradient, use the chain rule for (partial) derivatives and group together common terms,<br>
in a similar way we did for the gradient of the MSE loss (for linear regression) and the BCE loss (for logistic regression).

# Part 2: Coding Assignment (82 pts)

In this coding assignment, you will implement classifiers that use logistic regression to classify handwritten digits from the MNIST dataset.

The code cell below sets up the libraries and plotting settings for this analysis.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# make matplotlib figures appear inline in the notebook
%matplotlib inline
plt.rcParams['figure.figsize'] = (10.0, 8.0) # set default size of plots
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'

# Make the notebook automatically reload external python modules
%load_ext autoreload
%autoreload 2
# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

## 1. Loading and Preparing Data (5 pts)

We start by loading the MNIST data and adding random noise to every image.
Each sample in the MNIST dataset consists of 784 integer features in the range $[0,255]$ representing a $28 \times 28$ grayscale image.
The focus of most sections of this assignment is on binary classification between digits 6 and 8 (the last section extends this to multiple classes).
So we start by loading the subset of MNIST that contains images of digits 6 and 8 and applying some preprocessing.
Recall that before applying logistic regression training, we need to add a *bias feature* to each sample.

---

**Complete the function `add_bias_term` in `hw3.py`.** This function adds a constant 1 as the 0th feature to each sample.

---

The code cell below loads the data from file ``mnist_6n8.npz``,
normalizes the values in each feature to the range [0,1],
and adds random noise to each sample image.
It then displays random samples as bitmap images, and applies the function `add_bias_term` to the entire dataset.
Make sure the data file ``mnist_6n8.npz`` is in the working directory before executing this code.


In [ ]:
from hw3 import add_bias_term

NOISE_LEVEL = 0.7

# This function adds random noise to the given data (train or test).
def add_noise(X, noise_level):
    np.random.seed(42)
    return np.minimum(X + (np.random.rand(X.shape[0], X.shape[1]) < noise_level), 1)

# Load train and test data and normalize pixel values to [0, 1]
data = np.load('mnist_6n8.npz')

X_train = data["X_train"] / 255
X_test  = data["X_test"] / 255
y_train = data['y_train'].astype(int)
y_test  = data['y_test'].astype(int)

classes = np.unique(y_train)

# First, gradually add noise until NOISE_LEVEL to better understand the original data:
num_per_class = 6
noise_levels = np.linspace(0, NOISE_LEVEL, num=num_per_class)  # 0, 0.18, 0.36, 0.54, 0.72, 0.9

fig, axes = plt.subplots(2, num_per_class, figsize=(15, 5))
axes = axes.flatten()

for idx, noise in enumerate(noise_levels):
    # Add noise
    X_train_noisy = add_noise(X_train, noise)
    # Plot an example image for each class.
    for class_idx, class_label in enumerate(classes):
        # Find an index for the current class
        class_indices = np.where(y_train == class_label)[0]
        sample_idx = np.random.choice(class_indices)
        ax_pos = class_idx * num_per_class + idx
        axes[ax_pos].imshow(X_train_noisy[sample_idx].reshape(28, 28), cmap='gray')
        axes[ax_pos].set_title(f"Class {class_label}\nnoise={noise:.2f}", fontsize=9)
        axes[ax_pos].axis('off')
plt.suptitle(f"Effect of Noise on Example MNIST Images (Classes {classes[0]} and {classes[1]})", fontsize=14)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

## We now modify the data and add noise. The noisy data is more difficult to classify.

# Add random noise to train and test data.
X_train = add_noise(X_train, noise_level=NOISE_LEVEL)
X_test  = add_noise(X_test, noise_level=NOISE_LEVEL)

# Print basic stats.
print("Training set contains", X_train.shape[0], "samples with", X_train.shape[1], "features")
print("Test set contains", X_test.shape[0], "samples with", X_test.shape[1], "features")

# Plot 12 random images (6 for each class) from the training set, all with noise level = NOISE_LEVEL, in a 2x6 grid

fig, axes = plt.subplots(2, 6, figsize=(15, 5))
axes = axes.flatten()
for class_idx, class_label in enumerate(classes):
    # Find all indices for the current class
    class_indices = np.where(y_train == class_label)[0]
    # Randomly select num_per_class indices for this class
    selected_indices = np.random.choice(class_indices, size=num_per_class, replace=False)
    for i, idx in enumerate(selected_indices):
        ax = axes[class_idx * num_per_class + i]
        ax.imshow(X_train[idx].reshape(28, 28), cmap='gray')
        ax.set_title(f'Class {class_label}', fontsize=9)
        ax.axis('off')
plt.suptitle(f"Random MNIST Images (Classes {classes[0]} and {classes[1]}) with Noise Level {NOISE_LEVEL}", fontsize=14)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

# Add the bias term to all samples

X_train_with_bias = add_bias_term(X_train)
X_test_with_bias = add_bias_term(X_test)

# Verify that the size of each instance in the datasets has increased by 1
assert X_train_with_bias.shape[1] == X_test_with_bias.shape[1] == 785

## 2. Implementing The Logistic Regression Model (30 pts)


**Implement the class `LogisticRegressionGD` in `hw3.py` by completing the following four methods:**
* `predict_proba` - computes prediciton probabilities of the positive class for all samples in a dataset
* `predict` - computes predicted class labels for all samples in a dataset using a given probability threshold
* `BCE_loss` - computes the binary cross entropy (BCE) loss of a given dataset with true class labels
* `fit` - implements the gradient descent algorithm to find weights that minimize the BCE loss on a given labeled dataset

The constructor of this class is already implemented.
Read the documentation of each function carefully before you implement it.
In your implementation, assume that the dataset (`X`) already contains the added *bias feature*.
The class labels output by `predict` are the actual class labels (i.e. the digits 6 and 8) and not the binary values {0,1}.


Recall that the BCE loss is given by the following expression:
$$
\mathrm{BCE}(w;D) = -\frac{1}{n}\sum_{i=1}^n [y_i \log(\sigma(w^\top x^{(i)})) + (1-y_i)\log(1-\sigma(w^\top x^{(i)}))] ~,
$$

where:
* $\sigma(t) = \frac{1}{1 + e^{-t}}$ is the logistic function (a.k.a sigmoid function)
* $w$ is the weight vector (including $w_0$)
* $x^{(i)}$ is feature vector of the $i$-th sample (including the bias feature $x^{(i)}_0=1$)
* $y_i$ is the true class label of the $i$-th sample (after mapping to {0,1}})


The code cell below fits a logistic regression (LoR) model to the training dataset using standard gradient descent.

In [ ]:
from hw3 import LogisticRegressionGD

model = LogisticRegressionGD(learning_rate=0.001, max_iter=3000)

model.fit(X_train_with_bias, y_train)

import matplotlib.pyplot as plt

# Plot the BCE loss vs iteration number
plt.figure(figsize=(8,4))
plt.plot(model.loss_history_)
plt.xlabel('Iteration number')
plt.ylabel('BCE Loss')
plt.title('BCE Loss vs. Iteration Number')
plt.grid(True)
plt.show()

## 3. Minibatch Gradient Descent (12 pts)

If you ran the code in the previous seciton, you noticed that training takes a lot of time using standard gradient descent (GD).
We now use mini-batch GD to speed things up. 

---

**Implement the class `LogisticRegressionGD_MB` in `hw3.py` by completing the function `fit`.**
Because this class inherits `LogisticRegressionGD` from (2) above, this is the only function you need to implement for this class. 

---

The code cell below fits LoR models to the training set using mini-batch GD for 5 epochs with 128 samples per batch and using several values for the learning rate.
It then selects the learning rate that results in the smallest BCE loss after 5 epochs and trains a LoR model on the training set using this learning rate for 30 epochs. The weights of the final LoR model are plotted as a bitmap  image and this model is evaluated on the training and test data.

In [ ]:
from hw3 import LogisticRegressionGD_MB

# Fit LoR models using min-batch GD for 5 epochs with batch size 128 using variaous learning rates
n_epochs = 5
lo_learning_rates = [0.1, 0.05, 0.01]
loss_histories = {}
for learning_rate in lo_learning_rates:
    print(f"Training with learning rate: {learning_rate}")
    model_MB = LogisticRegressionGD_MB(learning_rate=learning_rate, eps=1e-9, random_state=1, batch_size=128, num_epochs=n_epochs)
    model_MB.fit(X_train_with_bias, y_train)
    loss_histories[learning_rate] = model_MB.loss_history_
    print(f"Loss after {n_epochs} epochs: {loss_histories[learning_rate][-1]}")

# Plot the BCE loss vs iteration number for each learning rate
plt.figure(figsize=(8,4))
for learning_rate, loss_history in loss_histories.items():
    plt.plot(loss_history, label=f'Learning Rate = {learning_rate}')
plt.xlabel('Iteration number')
plt.ylabel('BCE Loss')
plt.title('BCE Loss vs. Iteration Number')
plt.legend()
plt.grid(True)
plt.show()

# Select learning rate that leads to smallest BCE loss after 5 epochs
lo_bce_losses = [loss_histories[lr][-1] for lr in lo_learning_rates]
best_learning_rate = lo_learning_rates[np.argmin(lo_bce_losses)]
print(f"The learning rate attaining the minimal BCE is {best_learning_rate}. We pick it.")

# Train the model with the selected learning rate for 30 epochs
model = LogisticRegressionGD_MB(learning_rate=best_learning_rate, batch_size=128, num_epochs=30)
model.fit(X_train_with_bias, y_train)

# Convert the weights to a 28x28 image for visualization and plot them
weights = model.w_[1:].reshape(28, 28)
plt.figure(figsize=(4,4))
plt.imshow(weights)
plt.axis('off')
plt.title("weights of LoR classifier")
plt.show()

# Evaluate the model on the training and test sets
print(f"Train BCE: {model.BCE_loss(X_train_with_bias, y_train)}")
y_pred_train = model.predict(X_train_with_bias)
print(f"Train accuracy: {np.sum(y_pred_train == y_train) / len(y_train)}")

print(f"Test BCE: {model.BCE_loss(X_test_with_bias, y_test)}")
y_pred_test = model.predict(X_test_with_bias)
print(f"Test accuracy: {np.sum(y_pred_test == y_test) / len(y_test)}")

 ## 4. Evaluating The Classifier (10 pts)

**Complete the function `calc_metrics` in `hw3.py`.**
This function computes the following metrics used when evaluating a binary classifier.

Basic stats:
- **TP:** the number of positive samples that are correctly classified (as positive)
- **FP:** the number of negative samples that are        misclassified (as positive)
- **TN:** the number of negative samples that are correctly classified (as negative)
- **FN:** the number of positive samples that are        misclassified (as negative)

Simple rates:
- **TPR:** the fraction of   correctly predicted positive samples (TP) out of all positive samples (TP+FN); also called *Recall*
- **FPR:** the fraction of incorrectly predicted positive samples (FP) out of all negative samples (TN+FP); also called *Fall-out*
- **TNR:** the fraction of   correctly predicted negative samples (TN) out of all negative samples (TN+FP)
- **FNR:** the fraction of incorrectly predicted negative samples (FN) out of all positive samples (TP+FN)

Composite rates and metrics:
- **Accuracy:** the fraction of correctly predicted samples (positive or negative; TP+TN) out of all samples (TP+TN+FN+FP)
- **Risk:** the fraction of incorrectly predicted samples (positive or negative; FP+FN) out of all samples (TP+TN+FN+FP)
- **Precision:** the fraction of correctly predicted positive samples (TP) out of all samples **predicted as positive** (TP+FP);
  also called *Positive Predicted Value (PPV)*
- **F1** = 2 * (Precision * Recall) / (Precision + Recall) - a balanced measure combining precision and recall (TPR) by taking their harmonic mean

The code cell below applies this function to evaluate the LoR model you trained in the previous section on the training set and test set.
The evaluation considers digit 8 as the positive class and digit 6 as the negative class.

In [ ]:
# Calculate the metrics for the LogisticRegression classifier.
from hw3 import calc_metrics

# Predict the labels for the training and test sets
y_pred_train = model.predict(X_train_with_bias)
y_pred_test = model.predict(X_test_with_bias)

print(f"Training set (total = {y_train.shape[0]}):")
metrics_dic_train = calc_metrics(y_train, y_pred_train, positive_class=8)

print("Training set metrics:")
for key, value in metrics_dic_train.items():
    print(f"{key}: {value}")

print(f"\nTest set: (total = {y_test.shape[0]}):")
metrics_dic_test  = calc_metrics(y_test, y_pred_test, positive_class=8)

print("Test set metrics:")
for key, value in metrics_dic_test.items():
    print(f"{key}: {value}")



## 5. The Receiver Operating Characteristic (ROC) Curve (10 pts)

An important feature of (linear) binary classifiers is the possibility to tune their sensitivity to different kinds of errors.
In logistic regression models this can easily be achieved by tuning the probability threshold for positive predictions (which can be seen as considering hyperplanes parallel to the original one).
By default, we use a threshold of $\frac 1 2$ and associate a sample $x$ for which $\sigma(w^\top x)>\frac 1 2$ with positive prediction $\hat y = 1$. By increasing this threshold, we decrease the number of positive predictions which tends to decrease the number of false positives and increase the number of false negatives.

With the *receiver operating characteristic (ROC) curve* we examine the performance of a given binary classifier across different classification thresholds. For each threshold we plot the TPR (Y axis) against the FPR (X axis). This curve allows us to assess the classifier's performance across prediction thresholds. A common way to evaluate the classifier across all thresholds is using the area under the curve (AUC). This provides a single measure that tells us where a classifier falls in the spectrum between a perfect classifier (AUC=1) and a random one (AUC=$\frac 12$).

---

**Complete the function `fpr_tpr_per_threshold` in `hw3.py`.**
This function calculates the FPRs and TPRs of a given classifier across the probability thresholds in [0,1] in intervals of 0.01.

---

The code cell below applies function `fpr_tpr_per_threshold` to the trained classifier on the test set,
and uses the resulting TPR and FPR values to plot a ROC curve.
It then computes the AUC for this classifier and does the same for a random classifier.

In [ ]:
from hw3 import fpr_tpr_per_threshold

def illustrate_roc_curve_and_auc(fpr, tpr, label='ROC Curve'):
    """
    Plot the ROC curve and calculate AUC score for a classifier.

    Parameters
    ----------
    fpr : list or array
        False Positive Rate values.
    tpr : list or array
        True Positive Rate values.
    label : str, optional
        Name of the classifier to display in plot legend. Default is 'ROC Curve'.
    """

    # Compute the AUC score by integrating the ROC curve sorted by FPR.
    idcs = np.argsort(fpr)
    sorted_fpr = np.array(fpr)[idcs]
    sorted_tpr = np.array(tpr)[idcs]
    auc = np.sum((sorted_fpr[1:] - sorted_fpr[:-1]) * (sorted_tpr[1:] + sorted_tpr[:-1]) / 2)

    # Plot ROC curve
    plt.figure(figsize=(8, 6))
    plt.plot(sorted_fpr, sorted_tpr, label=f'{label} (AUC = {auc:.3f})')
    plt.plot([0, 1], [0, 1], 'k--', label='FPR = TPR')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curve')
    plt.legend()
    plt.grid(True)
    plt.show()


# Random classifier probabilities
seed = 1
np.random.seed(seed)
random_classification_probs = np.random.rand(len(y_test))
# Compute FPRs and TPRs for the random classifier
fpr , tpr = fpr_tpr_per_threshold(y_test, random_classification_probs, positive_class=8)
# Plot ROC curve and compute AUC score
illustrate_roc_curve_and_auc(fpr , tpr, label=f'Random Classifier (seed={seed})')
# this classifier is equivalent to a random guess, so its FPR and TPR are about equal and its AUC is 0.5

# Logistic regression classifier probabilities
logistic_classifier_probs = model.predict_proba(X_test_with_bias)
# Compute FPRs and TPRs for the logistic regression model
fpr , tpr = fpr_tpr_per_threshold(y_test, logistic_classifier_probs, positive_class=8)
# Plot ROC curve and compute AUC score
illustrate_roc_curve_and_auc(fpr , tpr, label=f'Logistic Regression')

## 6. One-vs-All Multi-Class Classification (15 pts)


We now extend the binary classifier to a multi-class setting using the one-vs-all (OvA) strategy.
In this approach, we first train a binary LoR model for each class separately, such that it distinguishes that class from all the other classes.
Multi-class prediction is then implemented by applying each LoR model to a data sample and selecting the class whose LoR model produces the highest (positive) prediciton probability.

**Implement the class `OneVsAllClassifier` in `hw3.py`, by completing the following three methods:**
* `fit`: train a collection of binary probabilistic classifiers (e.g., LoR) on a given dataset - one classifier per class. For each binary classifier, you should associate the positive label with one class and the negative label with all other classes. This should be done separately for each binary classifier.
* `predict_proba`: return a matrix of one-vs-all scores with shape `[n_samples, n_classes]`, where column `j` contains the positive-class probabilities from the binary classifier trained for the `j`-th class. The probabilities in each row of this matrix do not necessarily sum to 1!
* `predict`: predict the class using the OvA approach. 

 The `OneVsAllClassifier` class is a wrapper around a probabilistic binary classifier class `binary_classifier_class`.
 The default class for `binary_classifier_class` is `LogisticRegressionGD_MB` that you implemented in (3) above,
 but your implementation of class `OneVsAllClassifier` should not assume that this probabilistic classifier uses a LoR model.
 (We will see other probabilistic classifiers later in the course.)
 You should assume that `binary_classifier_class` has methods `predict_proba` and `fit`, and you should invoke them in your implementation.

---

The code cells below load data from file `mnist_6n8n9.npz` containing sample images for digits 6, 8, and 9.
The data is then processed using the same pipeline we applied above (including adding the bias feature),
a OvA LoR classifier is trained on the training dataset.
The resulting weight vectors are then plotted and the performance of the multi-class classifier is evaluated on the training and test sets.

In [ ]:
from hw3 import OneVsAllClassifier, LogisticRegressionGD_MB, add_bias_term

# Load the three-class dataset and prepare it in the same way as the binary dataset.
data_multi = np.load('mnist_6n8n9.npz')

X_train_multi = data_multi['X_train'] / 255
X_test_multi  = data_multi['X_test'] / 255
y_train_multi = data_multi['y_train'].astype(int)
y_test_multi  = data_multi['y_test'].astype(int)

X_train_multi = add_noise(X_train_multi, noise_level=NOISE_LEVEL)
X_test_multi  = add_noise(X_test_multi, noise_level=NOISE_LEVEL)

X_train_multi_with_bias = add_bias_term(X_train_multi)
X_test_multi_with_bias = add_bias_term(X_test_multi)
classes_multi = np.unique(y_train_multi)

print('Multi-class labels:', classes_multi)
print('Training set contains', X_train_multi_with_bias.shape[0], 'samples with', X_train_multi_with_bias.shape[1], 'features')
print('Test set contains', X_test_multi_with_bias.shape[0], 'samples with', X_test_multi_with_bias.shape[1], 'features')


In [ ]:
# Train one mini-batch logistic-regression classifier per class.
ova_model = OneVsAllClassifier(
    LogisticRegressionGD_MB,
    learning_rate=best_learning_rate,
    eps=1e-9,
    random_state=1,
    batch_size=128,
    num_epochs=30,
)
ova_model.fit(X_train_multi_with_bias, y_train_multi)

# Verify that predict_proba returns one score per class for every sample.
train_scores = ova_model.predict_proba(X_train_multi_with_bias)
test_scores = ova_model.predict_proba(X_test_multi_with_bias)
assert train_scores.shape == (X_train_multi_with_bias.shape[0], len(classes_multi))
assert test_scores.shape == (X_test_multi_with_bias.shape[0], len(classes_multi))

y_pred_train_multi = ova_model.predict(X_train_multi_with_bias)
y_pred_test_multi = ova_model.predict(X_test_multi_with_bias)

print(f'Train accuracy: {np.mean(y_pred_train_multi == y_train_multi)}')
print(f'Test accuracy: {np.mean(y_pred_test_multi == y_test_multi)}')


In [ ]:
# Visualize the learned weights for each one-vs-all classifier.
weights_per_class = [classifier.w_[1:].reshape(28, 28) for classifier in ova_model.classifiers_]
max_abs_weight = max(np.max(np.abs(weights)) for weights in weights_per_class)

fig, axes = plt.subplots(1, len(classes_multi), figsize=(4 * len(classes_multi), 4))
if len(classes_multi) == 1:
    axes = [axes]

for ax, class_label, weights in zip(axes, classes_multi, weights_per_class):
    im = ax.imshow(weights, cmap='coolwarm', vmin=-max_abs_weight, vmax=max_abs_weight)
    ax.set_title(f'Class {class_label} vs all')
    ax.axis('off')

fig.colorbar(im, ax=axes, fraction=0.025, pad=0.04)
plt.suptitle('One-vs-all classifier weights')
plt.show()


In [ ]:
# Build a multi-class confusion matrix. Rows are true classes and columns are predicted classes.
confusion = np.zeros((len(classes_multi), len(classes_multi)), dtype=int)
class_to_index = {label: i for i, label in enumerate(classes_multi)}

for true_label, pred_label in zip(y_test_multi, y_pred_test_multi):
    confusion[class_to_index[true_label], class_to_index[pred_label]] += 1

confusion_df = pd.DataFrame(
    confusion,
    index=[f'true {label}' for label in classes_multi],
    columns=[f'pred {label}' for label in classes_multi],
)
confusion_df
